# RNN 實戰：電影評論情感分析

> **項目目標**: 使用 LSTM/GRU 對 IMDB 電影評論進行正負面情感分類
> 
> **難度**: ⭐⭐⭐ 中級
> 
> **預計時間**: 2-3 小時

## 📋 項目大綱

1. 數據準備和探索
2. 文本預處理
3. 構建詞表
4. 模型設計（多種架構對比）
5. 訓練和評估
6. 模型解釋和可視化
7. 實際應用

---

## 1. 環境準備

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
from tqdm import tqdm

# 設置隨機種子
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

set_seed(42)

# 檢測設備
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用設備: {device}')

## 2. 數據加載和探索

我們使用 IMDB 電影評論數據集，包含 50,000 條評論（25k 訓練，25k 測試）

In [ ]:
# 使用 torchtext 加載 IMDB 數據集
from torchtext.datasets import IMDB
from torchtext.data.utils import get_tokenizer

# 如果沒有 torchtext，可以手動下載或使用示例數據
# 這裡提供一個示例數據生成函數
def generate_sample_data(num_samples=1000):
    """生成示例情感分析數據"""
    positive_words = ['good', 'great', 'excellent', 'amazing', 'wonderful', 'fantastic', 'love', 'perfect']
    negative_words = ['bad', 'terrible', 'awful', 'horrible', 'worst', 'hate', 'poor', 'disappointing']
    neutral_words = ['movie', 'film', 'story', 'actor', 'character', 'scene', 'plot', 'director']
    
    data = []
    for i in range(num_samples):
        label = np.random.randint(0, 2)
        if label == 1:  # 正面評論
            text = ' '.join(np.random.choice(positive_words + neutral_words, size=np.random.randint(10, 30)))
        else:  # 負面評論
            text = ' '.join(np.random.choice(negative_words + neutral_words, size=np.random.randint(10, 30)))
        data.append((text, label))
    
    return data

# 生成訓練和測試數據
train_data = generate_sample_data(5000)
test_data = generate_sample_data(1000)

print(f"訓練集大小: {len(train_data)}")
print(f"測試集大小: {len(test_data)}")
print(f"\n示例數據:")
for i in range(3):
    text, label = train_data[i]
    print(f"  文本: {text[:50]}...")
    print(f"  標籤: {'正面' if label == 1 else '負面'}\n")

## 3. 文本預處理

包括：
1. 小寫化
2. 去除標點符號
3. 分詞
4. 構建詞表
5. 文本轉索引

In [ ]:
class Vocabulary:
    """詞表類"""
    def __init__(self, max_size=10000, min_freq=2):
        self.max_size = max_size
        self.min_freq = min_freq
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}
        self.word_freq = Counter()
    
    def build_vocab(self, texts):
        """從文本列表構建詞表"""
        for text in texts:
            tokens = self.tokenize(text)
            self.word_freq.update(tokens)
        
        # 按頻率排序，取前 max_size 個詞
        for word, freq in self.word_freq.most_common(self.max_size - 2):
            if freq >= self.min_freq:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx] = word
        
        print(f"詞表大小: {len(self.word2idx)}")
    
    def tokenize(self, text):
        """簡單的分詞函數"""
        text = text.lower()
        text = re.sub(r'[^a-z0-9\s]', '', text)
        return text.split()
    
    def encode(self, text):
        """將文本轉換為索引序列"""
        tokens = self.tokenize(text)
        return [self.word2idx.get(word, self.word2idx['<UNK>']) for word in tokens]
    
    def decode(self, indices):
        """將索引序列轉換回文本"""
        return ' '.join([self.idx2word.get(idx, '<UNK>') for idx in indices])
    
    def __len__(self):
        return len(self.word2idx)

# 構建詞表
vocab = Vocabulary(max_size=10000, min_freq=2)
vocab.build_vocab([text for text, _ in train_data])

# 測試編碼和解碼
sample_text = train_data[0][0]
encoded = vocab.encode(sample_text)
decoded = vocab.decode(encoded)
print(f"\n原始文本: {sample_text}")
print(f"編碼結果: {encoded[:10]}...")
print(f"解碼結果: {decoded}")

## 4. 數據集和 DataLoader

In [ ]:
class SentimentDataset(Dataset):
    """情感分析數據集"""
    def __init__(self, data, vocab):
        self.data = data
        self.vocab = vocab
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        text, label = self.data[idx]
        encoded = self.vocab.encode(text)
        return torch.tensor(encoded), torch.tensor(label)

def collate_fn(batch):
    """自定義批處理函數（處理變長序列）"""
    texts, labels = zip(*batch)
    lengths = torch.tensor([len(text) for text in texts])
    
    # Padding
    padded_texts = pad_sequence(texts, batch_first=True, padding_value=0)
    labels = torch.stack(labels)
    
    return padded_texts, labels, lengths

# 創建數據集和 DataLoader
train_dataset = SentimentDataset(train_data, vocab)
test_dataset = SentimentDataset(test_data, vocab)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)

# 測試 DataLoader
for texts, labels, lengths in train_loader:
    print(f"Batch 形狀:")
    print(f"  texts: {texts.shape}")
    print(f"  labels: {labels.shape}")
    print(f"  lengths: {lengths.shape}")
    print(f"  最大長度: {lengths.max().item()}")
    print(f"  最小長度: {lengths.min().item()}")
    break

## 5. 模型架構

我們實現三種模型進行對比：
1. 簡單 LSTM
2. 雙向 LSTM
3. 多層雙向 LSTM + Attention

In [ ]:
class SimpleLSTM(nn.Module):
    """簡單的 LSTM 情感分類器"""
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes=2, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x, lengths):
        # x: (batch, seq_len)
        embedded = self.dropout(self.embedding(x))
        
        # LSTM
        lstm_out, (h, c) = self.lstm(embedded)
        
        # 使用最後一個時間步的隱狀態
        h = h.squeeze(0)
        h = self.dropout(h)
        
        return self.fc(h)


class BiLSTM(nn.Module):
    """雙向 LSTM 情感分類器"""
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes=2, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.bilstm = nn.LSTM(embed_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
    
    def forward(self, x, lengths):
        embedded = self.dropout(self.embedding(x))
        
        # 雙向 LSTM
        lstm_out, (h, c) = self.bilstm(embedded)
        
        # 拼接前向和後向的最後隱狀態
        h_fwd = h[-2, :, :]
        h_bwd = h[-1, :, :]
        h = torch.cat([h_fwd, h_bwd], dim=1)
        h = self.dropout(h)
        
        return self.fc(h)


class AttentionBiLSTM(nn.Module):
    """帶注意力機制的雙向 LSTM"""
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes=2, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.bilstm = nn.LSTM(embed_dim, hidden_dim, num_layers=2,
                             bidirectional=True, dropout=dropout, batch_first=True)
        self.attention = nn.Linear(hidden_dim * 2, 1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
    
    def forward(self, x, lengths):
        embedded = self.dropout(self.embedding(x))
        
        # 雙向 LSTM
        lstm_out, _ = self.bilstm(embedded)
        
        # 注意力機制
        attn_weights = torch.softmax(self.attention(lstm_out), dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        context = self.dropout(context)
        
        return self.fc(context)


# 創建模型
vocab_size = len(vocab)
embed_dim = 100
hidden_dim = 128

model = AttentionBiLSTM(vocab_size, embed_dim, hidden_dim).to(device)
print(f"\n模型架構:")
print(model)
print(f"\n參數量: {sum(p.numel() for p in model.parameters()):,}")

## 6. 訓練設置

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """訓練一個 epoch"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for texts, labels, lengths in tqdm(dataloader, desc="Training"):
        texts, labels = texts.to(device), labels.to(device)
        
        # 前向傳播
        outputs = model(texts, lengths)
        loss = criterion(outputs, labels)
        
        # 反向傳播
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        # 統計
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return total_loss / len(dataloader), 100. * correct / total


def evaluate(model, dataloader, criterion, device):
    """評估模型"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for texts, labels, lengths in dataloader:
            texts, labels = texts.to(device), labels.to(device)
            
            outputs = model(texts, lengths)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    return total_loss / len(dataloader), 100. * correct / total


# 訓練設置
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

num_epochs = 10
best_acc = 0
train_losses, train_accs = [], []
test_losses, test_accs = [], []

## 7. 開始訓練

In [ ]:
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 50)
    
    # 訓練
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    # 評估
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    test_losses.append(test_loss)
    test_accs.append(test_acc)
    
    # 學習率調整
    scheduler.step(test_loss)
    
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%")
    
    # 保存最佳模型
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"✓ 最佳模型已保存！準確率: {best_acc:.2f}%")

## 8. 訓練結果可視化

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# 損失曲線
ax1.plot(train_losses, label='Train Loss', marker='o')
ax1.plot(test_losses, label='Test Loss', marker='s')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Test Loss')
ax1.legend()
ax1.grid(True)

# 準確率曲線
ax2.plot(train_accs, label='Train Acc', marker='o')
ax2.plot(test_accs, label='Test Acc', marker='s')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training and Test Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n最終結果:")
print(f"  最佳測試準確率: {best_acc:.2f}%")
print(f"  最終訓練準確率: {train_accs[-1]:.2f}%")
print(f"  最終測試準確率: {test_accs[-1]:.2f}%")

## 9. 模型預測和解釋

In [ ]:
def predict_sentiment(text, model, vocab, device):
    """預測單條文本的情感"""
    model.eval()
    
    # 編碼文本
    encoded = vocab.encode(text)
    tensor = torch.tensor(encoded).unsqueeze(0).to(device)
    length = torch.tensor([len(encoded)])
    
    # 預測
    with torch.no_grad():
        output = model(tensor, length)
        probs = torch.softmax(output, dim=1)
        pred = output.argmax(1).item()
    
    sentiment = "正面 😊" if pred == 1 else "負面 😞"
    confidence = probs[0][pred].item() * 100
    
    return sentiment, confidence, probs[0].cpu().numpy()


# 測試預測
test_texts = [
    "This movie is absolutely fantastic! I loved every minute of it.",
    "Terrible film, waste of time and money. Very disappointing.",
    "The acting was good but the plot was confusing.",
]

print("\n情感預測測試:")
print("=" * 80)
for text in test_texts:
    sentiment, confidence, probs = predict_sentiment(text, model, vocab, device)
    print(f"\n文本: {text}")
    print(f"預測: {sentiment} (置信度: {confidence:.2f}%)")
    print(f"概率分佈: 負面={probs[0]:.3f}, 正面={probs[1]:.3f}")
    print("-" * 80)

## 10. 注意力權重可視化（如果使用 AttentionBiLSTM）

In [ ]:
def visualize_attention(text, model, vocab, device):
    """可視化注意力權重"""
    model.eval()
    
    # 編碼
    tokens = vocab.tokenize(text)
    encoded = vocab.encode(text)
    tensor = torch.tensor(encoded).unsqueeze(0).to(device)
    
    # 獲取注意力權重
    with torch.no_grad():
        embedded = model.embedding(tensor)
        lstm_out, _ = model.bilstm(embedded)
        attn_weights = torch.softmax(model.attention(lstm_out), dim=1)
    
    attn_weights = attn_weights.squeeze().cpu().numpy()
    
    # 繪圖
    plt.figure(figsize=(12, 3))
    plt.bar(range(len(tokens)), attn_weights)
    plt.xticks(range(len(tokens)), tokens, rotation=45, ha='right')
    plt.ylabel('Attention Weight')
    plt.title('Attention Weights Visualization')
    plt.tight_layout()
    plt.show()
    
    # 打印重要詞語
    important_words = sorted(zip(tokens, attn_weights), key=lambda x: x[1], reverse=True)[:5]
    print("\n最重要的詞語:")
    for word, weight in important_words:
        print(f"  {word}: {weight:.4f}")

# 如果模型是 AttentionBiLSTM，執行注意力可視化
if isinstance(model, AttentionBiLSTM):
    sample_text = "This movie is absolutely fantastic and amazing"
    visualize_attention(sample_text, model, vocab, device)

## 11. 模型對比實驗

In [ ]:
def compare_models():
    """對比不同模型的性能"""
    models = {
        'Simple LSTM': SimpleLSTM(vocab_size, embed_dim, hidden_dim),
        'BiLSTM': BiLSTM(vocab_size, embed_dim, hidden_dim),
        'Attention BiLSTM': AttentionBiLSTM(vocab_size, embed_dim, hidden_dim)
    }
    
    results = {}
    
    for name, model in models.items():
        print(f"\n訓練 {name}...")
        model = model.to(device)
        optimizer = optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.CrossEntropyLoss()
        
        best_acc = 0
        for epoch in range(5):  # 快速訓練 5 個 epoch
            train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
            test_loss, test_acc = evaluate(model, test_loader, criterion, device)
            
            if test_acc > best_acc:
                best_acc = test_acc
        
        results[name] = {
            'params': sum(p.numel() for p in model.parameters()),
            'best_acc': best_acc
        }
        
        print(f"{name} - 最佳準確率: {best_acc:.2f}%")
    
    # 可視化對比
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    names = list(results.keys())
    params = [results[n]['params'] for n in names]
    accs = [results[n]['best_acc'] for n in names]
    
    ax1.bar(names, params)
    ax1.set_ylabel('Number of Parameters')
    ax1.set_title('Model Complexity')
    ax1.tick_params(axis='x', rotation=15)
    
    ax2.bar(names, accs)
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('Model Performance')
    ax2.tick_params(axis='x', rotation=15)
    
    plt.tight_layout()
    plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

# 執行模型對比（可選，耗時較長）
# compare_models()

## 12. 互動式預測界面

In [ ]:
def interactive_prediction():
    """互動式情感預測"""
    print("\n=" * 80)
    print("情感分析互動式預測系統")
    print("=" * 80)
    print("輸入 'quit' 退出\n")
    
    while True:
        text = input("\n請輸入電影評論: ").strip()
        
        if text.lower() == 'quit':
            print("感謝使用！再見 👋")
            break
        
        if not text:
            print("請輸入有效的文本")
            continue
        
        sentiment, confidence, probs = predict_sentiment(text, model, vocab, device)
        
        print(f"\n預測結果: {sentiment}")
        print(f"置信度: {confidence:.2f}%")
        print(f"詳細概率: 負面={probs[0]:.3f}, 正面={probs[1]:.3f}")
        print("-" * 80)

# 運行互動式預測
# interactive_prediction()

## 13. 模型部署準備

In [ ]:
# 保存完整模型（用於部署）
import pickle

# 保存模型和詞表
torch.save({
    'model_state_dict': model.state_dict(),
    'vocab': vocab,
    'config': {
        'vocab_size': vocab_size,
        'embed_dim': embed_dim,
        'hidden_dim': hidden_dim
    }
}, 'sentiment_model_complete.pth')

print("✓ 模型已保存到 sentiment_model_complete.pth")

# 加載模型示例
def load_model(path):
    checkpoint = torch.load(path)
    vocab = checkpoint['vocab']
    config = checkpoint['config']
    
    model = AttentionBiLSTM(
        config['vocab_size'],
        config['embed_dim'],
        config['hidden_dim']
    )
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    return model, vocab

# 測試加載
# loaded_model, loaded_vocab = load_model('sentiment_model_complete.pth')
# print("✓ 模型加載成功！")

## 14. 總結和改進建議

### 本項目學到的內容

1. ✅ 完整的文本分類流程
2. ✅ LSTM/GRU 在 NLP 中的應用
3. ✅ 注意力機制的實現
4. ✅ 模型訓練和評估
5. ✅ 可視化和解釋

### 性能改進建議

1. **使用預訓練詞嵌入**
   ```python
   # 使用 GloVe 或 Word2Vec
   embedding = nn.Embedding.from_pretrained(pretrained_embeddings)
   ```

2. **數據增強**
   - 同義詞替換
   - 回譯（Back-translation）
   - 隨機插入/刪除

3. **更複雜的架構**
   - CNN-LSTM 組合
   - Transformer 模型
   - BERT fine-tuning

4. **超參數優化**
   - 使用 Optuna 或 Ray Tune
   - Grid Search / Random Search

### 延伸項目

- 多分類情感分析（1-5 星評分）
- 方面級情感分析（Aspect-based Sentiment Analysis）
- 跨域情感分析（Domain Adaptation）
- 多語言情感分析

---

**恭喜完成這個項目！🎉**
